# 🔬 Model Pipeline: Qwen-2.5-7B-Instruct (4-bit QLoRA)
### **Jadavpur University Research Internship | Under: Prof. Tohida Rehman**
**Track:** FIRE MixSub-SciHigh Scientific Highlight Summarization
**Model ID:** `Qwen/Qwen2.5-7B-Instruct`
**Architecture:** CAUSAL (4-bit QLoRA)

---
### 🎯 Strict Uniform Benchmark Constraints (Scientifically Fair)
- **`MAX_SOURCE_LENGTH = 512` tokens**: Uniform input length across all 6 models.
- **`MAX_TARGET_LENGTH = 128` tokens**: Uniform output generation limit across all 6 models.

### 📊 The 3 Non-Redundant Benchmark Pillars Measured:
1. **Overall Quality**: ROUGE-L F1 (↑ Higher is Better) & Target F1 Score (↑)
2. **Hallucination Risk**: Ungrounded Token % (↓ Lower is Better) & Precision Source (% grounded in abstract)
3. **Inference Time**: Wall-Clock Latency per Sample (↓ Lower is Faster) & Throughput (tok/s ↑)

In [ ]:
# Step 1: Mount Google Drive for Persistent Checkpoint Resuming
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE_DIR = '/content/drive/MyDrive/JU_Benchmark_Project'
MODEL_CHECKPOINT_DIR = os.path.join(DRIVE_BASE_DIR, 'checkpoints', 'qwen-2.5-7b')
RESULTS_DIR = os.path.join(DRIVE_BASE_DIR, 'results')
DATASET_DIR = os.path.join(DRIVE_BASE_DIR, 'Dataset')

os.makedirs(MODEL_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

# Uniform Token Limits Across All Models
MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128

print('✅ Drive mounted!')
print('📁 Checkpoint directory:', MODEL_CHECKPOINT_DIR)
print('📁 Results directory:   ', RESULTS_DIR)
print(f'📏 Uniform Token Limits: Input={MAX_SOURCE_LENGTH} | Output={MAX_TARGET_LENGTH}')


In [ ]:
# Step 2: Install Libraries
!pip install -q --upgrade pip
!pip install -q transformers peft bitsandbytes datasets accelerate evaluate rouge-score tabulate

import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


In [ ]:
# Step 3: Verify Dataset Files
import pandas as pd

TRAIN_CSV = os.path.join(DATASET_DIR, 'MixSub-SciHigh_train_FIRE.csv')
VAL_CSV = os.path.join(DATASET_DIR, 'MixSub-SciHigh_val_FIRE.csv')
TEST_CSV = os.path.join(DATASET_DIR, 'MixSub-SciHigh_test_FIRE(Masked).csv')

for p, name in [(TRAIN_CSV, 'Train'), (VAL_CSV, 'Val'), (TEST_CSV, 'Test')]:
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(f'✅ {name} set ready: {len(df):,} rows')
    else:
        print(f'⚠️ Warning: {p} not found in Drive. Please upload to {DATASET_DIR}')


In [ ]:
# Step 4: Core Metric Calculations (3 Pillars: Quality, Hallucination Risk, Latency)
import re
from collections import Counter
import numpy as np
from rouge_score import rouge_scorer

def clean_tokenize(text):
    if not isinstance(text, str):
        return []
    return re.findall(r'\b[a-zA-Z0-9]+(?:\'[a-zA-Z0-9]+)?\b', text.lower())

def get_ngrams(tokens, n=1):
    if len(tokens) < n or n < 1:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

def compute_pillars_for_sample(gen_text, src_text, tgt_text):
    gen_tokens = clean_tokenize(gen_text)
    src_tokens = clean_tokenize(src_text)
    tgt_tokens = clean_tokenize(tgt_text)

    gen_counts = Counter(get_ngrams(gen_tokens, 1))
    src_counts = Counter(get_ngrams(src_tokens, 1))
    tgt_counts = Counter(get_ngrams(tgt_tokens, 1))

    total_gen = sum(gen_counts.values())
    total_tgt = sum(tgt_counts.values())

    # Pillar 2: Factuality / Hallucination Risk (Tokens ungrounded in Source Abstract)
    overlap_src = sum((gen_counts & src_counts).values())
    precision_src = (overlap_src / total_gen) if total_gen > 0 else 0.0
    hallucination_risk = 1.0 - precision_src # Ungrounded fraction (Lower is Better)

    # Pillar 1: Target Highlight Alignment
    overlap_tgt = sum((gen_counts & tgt_counts).values())
    precision_tgt = (overlap_tgt / total_gen) if total_gen > 0 else 0.0
    recall_tgt = (overlap_tgt / total_tgt) if total_tgt > 0 else 0.0
    f1_tgt = (2 * precision_tgt * recall_tgt / (precision_tgt + recall_tgt)) if (precision_tgt + recall_tgt) > 0 else 0.0

    return {
        'hallucination_risk': hallucination_risk,
        'precision_src': precision_src,
        'precision_tgt': precision_tgt,
        'recall_tgt': recall_tgt,
        'f1_tgt': f1_tgt
    }

def evaluate_core_benchmarks(predictions, sources, references, latencies=None, generated_tokens=None):
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rl_f1_list, rl_p_list, rl_r_list = [], [], []

    for pred, ref in zip(predictions, references):
        score = scorer.score(ref, pred)
        rl_f1_list.append(score['rougeL'].fmeasure)
        rl_p_list.append(score['rougeL'].precision)
        rl_r_list.append(score['rougeL'].recall)

    sample_metrics = [compute_pillars_for_sample(p, s, r) for p, s, r in zip(predictions, sources, references)]

    results = {
        # Pillar 1: Overall Quality (↑ Higher is Better)
        'overall_quality_rougeL_f1': float(np.mean(rl_f1_list)),
        'rougeL_precision': float(np.mean(rl_p_list)),
        'rougeL_recall': float(np.mean(rl_r_list)),
        'f1_score_target': float(np.mean([x['f1_tgt'] for x in sample_metrics])),
        'precision_target': float(np.mean([x['precision_tgt'] for x in sample_metrics])),
        'recall_target': float(np.mean([x['recall_tgt'] for x in sample_metrics])),

        # Pillar 2: Hallucination Risk (↓ Lower is Better)
        'hallucination_risk_pct': float(np.mean([x['hallucination_risk'] for x in sample_metrics])),
        'source_grounding_precision': float(np.mean([x['precision_src'] for x in sample_metrics])),
    }

    # Pillar 3: Inference Time & Speed
    if latencies is not None and len(latencies) > 0:
        results['wall_clock_latency_sec'] = float(np.mean(latencies)) # ↓ Lower is Better
        results['total_wall_clock_sec'] = float(np.sum(latencies))

    if generated_tokens is not None and latencies is not None and len(latencies) > 0:
        tokens_per_sec = [t / max(l, 1e-4) for t, l in zip(generated_tokens, latencies)]
        results['throughput_tokens_per_sec'] = float(np.mean(tokens_per_sec)) # ↑ Higher is Better

    return results

print('✅ 3-Pillar Benchmark Engine Loaded!')


In [ ]:
# Step 5: Train & Optimize Qwen-2.5-7B-Instruct (4-bit QLoRA)
HF_TOKEN = '' # HuggingFace Token for gated models
NUM_EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 8
SAVE_STEPS = 100

import gc
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

compute_dtype = torch.float16 if not torch.cuda.is_bf16_supported() else torch.bfloat16
device_map = {'': 0} if torch.cuda.is_available() else None

train_df = pd.read_csv(TRAIN_CSV).dropna(subset=['Abstract', 'Highlights'])
val_df = pd.read_csv(VAL_CSV).dropna(subset=['Abstract', 'Highlights'])

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-7B-Instruct', token=HF_TOKEN or None, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# Preprocess with Strict Token Limits
def prep(batch):
    texts = [f'<|im_start|>system\nYou are an expert scientific researcher. Extract concise, factual research highlights.<|im_end|>\n<|im_start|>user\nAbstract:\n{a.strip()}<|im_end|>\n<|im_start|>assistant\n{h.strip()}<|im_end|>' for a, h in zip(batch['Abstract'], batch['Highlights'])]
    tok = tokenizer(texts, max_length=MAX_SOURCE_LENGTH + MAX_TARGET_LENGTH, truncation=True)
    tok['labels'] = tok['input_ids'].copy()
    return tok
raw_train_ds = Dataset.from_pandas(train_df, preserve_index=False)
train_ds = raw_train_ds.map(prep, batched=True, remove_columns=raw_train_ds.column_names)
raw_val_ds = Dataset.from_pandas(val_df, preserve_index=False)
val_ds = raw_val_ds.map(prep, batched=True, remove_columns=raw_val_ds.column_names)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True
)
model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-7B-Instruct',
    quantization_config=bnb_config,
    device_map=device_map,
    torch_dtype=compute_dtype,
    trust_remote_code=True,
    token=HF_TOKEN or None
)
model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none'
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir=MODEL_CHECKPOINT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=2e-4,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=10,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    fp16=True if not torch.cuda.is_bf16_supported() else False,
    bf16=True if torch.cuda.is_bf16_supported() else False,
    optim='paged_adamw_8bit' if True else 'adamw_torch',
    report_to='none'
)
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds, tokenizer=tokenizer, data_collator=data_collator)
# Checkpoint Resuming Check
resume_ckpt = None
if os.path.exists(MODEL_CHECKPOINT_DIR):
    ckpts = [os.path.join(MODEL_CHECKPOINT_DIR, d) for d in os.listdir(MODEL_CHECKPOINT_DIR) if d.startswith('checkpoint-') and os.path.isdir(os.path.join(MODEL_CHECKPOINT_DIR, d))]
    if ckpts:
        ckpts.sort(key=lambda x: int(x.split('-')[-1]))
        resume_ckpt = ckpts[-1]
        print('--> 🔄 Found previous checkpoint on Google Drive! Resuming from:', resume_ckpt)

print('Starting training...')
trainer.train(resume_from_checkpoint=resume_ckpt)

final_adapter = os.path.join(MODEL_CHECKPOINT_DIR, 'final_adapter')
trainer.save_model(final_adapter)
tokenizer.save_pretrained(final_adapter)
print('✅ Fine-tuned adapter saved to Drive:', final_adapter)

del model, trainer
torch.cuda.empty_cache()
gc.collect()


In [ ]:
# Step 6: Benchmarking & Test Predictions for Qwen-2.5-7B-Instruct (4-bit QLoRA)
MAX_EVAL_SAMPLES = 100 # Validation samples to evaluate

import time
import json
from peft import PeftModel

eval_val_df = pd.read_csv(VAL_CSV).dropna(subset=['Abstract', 'Highlights'])
if MAX_EVAL_SAMPLES and len(eval_val_df) > MAX_EVAL_SAMPLES:
    eval_val_df = eval_val_df.iloc[:MAX_EVAL_SAMPLES]

test_df = pd.read_csv(TEST_CSV)
test_df['Abstract'] = test_df['Abstract'].fillna('')

final_adapter = os.path.join(MODEL_CHECKPOINT_DIR, 'final_adapter')
tokenizer = AutoTokenizer.from_pretrained(final_adapter if os.path.exists(final_adapter) else 'Qwen/Qwen2.5-7B-Instruct', token=HF_TOKEN or None, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_m = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-7B-Instruct',
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=compute_dtype),
    device_map=device_map,
    torch_dtype=compute_dtype,
    token=HF_TOKEN or None,
    trust_remote_code=True
)
if os.path.exists(final_adapter):
    print('--> Loading fine-tuned adapter weights from Drive...')
    eval_model = PeftModel.from_pretrained(base_m, final_adapter)
else:
    print('--> No fine-tuned adapter found, evaluating base model...')
    eval_model = base_m

eval_model.eval()

# GPU Warmup
with torch.no_grad():
    w_in = tokenizer(f'<|im_start|>system\nYou are an expert scientific researcher. Extract concise, factual research highlights.<|im_end|>\n<|im_start|>user\nAbstract:\n{"warmup"}<|im_end|>\n<|im_start|>assistant\n', return_tensors='pt').to(eval_model.device)
    _ = eval_model.generate(**w_in, max_new_tokens=10)

predictions, latencies, token_counts = [], [], []
sources = eval_val_df['Abstract'].tolist()
references = eval_val_df['Highlights'].tolist()

print(f'Benchmarking {len(eval_val_df)} validation samples (Uniform Input={MAX_SOURCE_LENGTH}, Output={MAX_TARGET_LENGTH})...')
for i, abs_text in enumerate(sources):
    prompt = f'<|im_start|>system\nYou are an expert scientific researcher. Extract concise, factual research highlights.<|im_end|>\n<|im_start|>user\nAbstract:\n{abs_text.strip()}<|im_end|>\n<|im_start|>assistant\n'

    in_t = tokenizer(prompt, max_length=MAX_SOURCE_LENGTH, truncation=True, return_tensors='pt').to(eval_model.device)
    p_len = in_t['input_ids'].shape[1]

    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t0 = time.perf_counter()
    with torch.no_grad():
        out = eval_model.generate(
            **in_t,
            max_new_tokens=MAX_TARGET_LENGTH,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    elapsed = time.perf_counter() - t0
    latencies.append(elapsed)

    new_toks = out[0][p_len:]
    pred_text = tokenizer.decode(new_toks, skip_special_tokens=True).strip()
    n_toks = len(new_toks)
    predictions.append(pred_text)
    token_counts.append(n_toks)
    if (i + 1) % 25 == 0 or (i + 1) == len(eval_val_df):
        print(f'  Sample {i+1}/{len(eval_val_df)} | Latency: {elapsed:.2f}s ({n_toks} tokens)')

# Compute 3-Pillar Benchmarks
bench_res = evaluate_core_benchmarks(predictions, sources, references, latencies, token_counts)
bench_res['model'] = 'qwen-2.5-7b'
bench_res['model_id'] = 'Qwen/Qwen2.5-7B-Instruct'
bench_res['max_source_length'] = MAX_SOURCE_LENGTH
bench_res['max_target_length'] = MAX_TARGET_LENGTH

res_file = os.path.join(RESULTS_DIR, 'benchmark_qwen-2.5-7b.json')
with open(res_file, 'w') as f:
    json.dump(bench_res, f, indent=4)
print('✅ 3-Pillar Benchmark saved to:', res_file)

# Generate Masked Test Predictions
print(f'Generating test highlights for {len(test_df)} masked samples...')
test_preds = []
for idx, abs_text in enumerate(test_df['Abstract']):
    prompt = f'<|im_start|>system\nYou are an expert scientific researcher. Extract concise, factual research highlights.<|im_end|>\n<|im_start|>user\nAbstract:\n{abs_text.strip()}<|im_end|>\n<|im_start|>assistant\n'

    in_t = tokenizer(prompt, max_length=MAX_SOURCE_LENGTH, truncation=True, return_tensors='pt').to(eval_model.device)
    p_len = in_t['input_ids'].shape[1]
    with torch.no_grad():
        out = eval_model.generate(**in_t, max_new_tokens=MAX_TARGET_LENGTH, do_sample=False, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    pred_str = tokenizer.decode(out[0][p_len:], skip_special_tokens=True).strip()
    test_preds.append(pred_str)
    if (idx + 1) % 100 == 0 or (idx + 1) == len(test_df):
        print(f'  Test progress: {idx+1}/{len(test_df)}')

test_sub_df = test_df.copy()
test_sub_df['Highlights'] = test_preds
test_sub_csv = os.path.join(RESULTS_DIR, 'MixSub-SciHigh_test_predictions_qwen-2.5-7b.csv')
test_sub_df.to_csv(test_sub_csv, index=False)
print('✅ Masked test predictions saved to:', test_sub_csv)

del eval_model, base_m
torch.cuda.empty_cache()
gc.collect()
